# 10 - Multi-Target PubMed Evidence Exploration

This notebook extends the multi-target pipeline from ChEMBL into PubMed literature evidence.

Input:

```text
data/processed/multi_target_drug_recommendations.csv
```

Outputs:

```text
data/raw/pubmed/multi_target_pubmed_search_raw.json
data/raw/pubmed/multi_target_pubmed_summary_raw.json
data/processed/multi_target_pubmed_evidence.csv
data/processed/multi_target_pubmed_summary.csv
data/processed/multi_target_pubmed_coverage_summary.csv
```

The notebook is intentionally exploratory. It limits the number of drugs searched per target so we can check coverage and query quality before scaling to all target-drug pairs.

## 1. Setup

In [1]:
from pathlib import Path
import json
import re
import time

import pandas as pd
import requests

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw" / "pubmed"
PROCESSED_DIR = DATA_DIR / "processed"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Raw PubMed folder:", RAW_DIR)
print("Processed folder:", PROCESSED_DIR)

Project root: /Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/ai-precision-medicine-lab/therapeutic-strategy-assistant
Raw PubMed folder: /Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/ai-precision-medicine-lab/therapeutic-strategy-assistant/data/raw/pubmed
Processed folder: /Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/ai-precision-medicine-lab/therapeutic-strategy-assistant/data/processed


## 2. Load Multi-Target Drug Recommendations

In [2]:
recommendations_file = PROCESSED_DIR / "multi_target_drug_recommendations.csv"

if not recommendations_file.exists():
    raise FileNotFoundError(
        f"Missing {recommendations_file}. Run notebooks/09_multi_target_chembl_exploration.ipynb first."
    )

recommendations_df = pd.read_csv(recommendations_file)

print("Rows:", len(recommendations_df))
print("Targets:", recommendations_df["target_symbol"].nunique())
print(recommendations_df.groupby("target_symbol").size().sort_values(ascending=False).to_string())

display(recommendations_df.head(20))

Rows: 207
Targets: 8
target_symbol
EGFR      79
ERBB2     40
MET       39
BRAF      14
VEGFA     13
ALK       11
PIK3CA     9
KRAS       2


,target_symbol,target_display_name,target_full_name,target_chembl_id,target_pref_name,drug_name,molecule_chembl_id,molecule_type,action_type,mechanism_of_action,approval_status,max_phase,first_approval
0,ALK,ALK,ALK tyrosine kinase receptor,CHEMBL4247,ALK tyrosine kinase receptor,ALECTINIB HYDROCHLORIDE,CHEMBL3707320,Small molecule,INHIBITOR,ALK tyrosine kinase receptor inhibitor,Approved,4.0,2015.0
1,ALK,ALK,ALK tyrosine kinase receptor,CHEMBL4247,ALK tyrosine kinase receptor,ASP-3026,CHEMBL3545360,Small molecule,INHIBITOR,ALK tyrosine kinase receptor inhibitor,Investigational (Phase 1),1.0,NaN
2,ALK,ALK,ALK tyrosine kinase receptor,CHEMBL4247,ALK tyrosine kinase receptor,BRIGATINIB,CHEMBL3545311,Small molecule,INHIBITOR,ALK tyrosine kinase receptor inhibitor,Approved,4.0,2017.0
3,ALK,ALK,ALK tyrosine kinase receptor,CHEMBL4247,ALK tyrosine kinase receptor,CEP-37440,CHEMBL3951811,Small molecule,INHIBITOR,ALK tyrosine kinase receptor inhibitor,Investigational (Phase 1),1.0,NaN
4,ALK,ALK,ALK tyrosine kinase receptor,CHEMBL4247,ALK tyrosine kinase receptor,CERITINIB,CHEMBL2403108,Small molecule,INHIBITOR,ALK tyrosine kinase receptor inhibitor,Approved,4.0,2014.0
5,ALK,ALK,ALK tyrosine kinase receptor,CHEMBL4247,ALK tyrosine kinase receptor,CONTELTINIB,CHEMBL3899477,Small molecule,INHIBITOR,ALK tyrosine kinase receptor inhibitor,Investigational (Phase 1),1.0,NaN
6,ALK,ALK,ALK tyrosine kinase receptor,CHEMBL4247,ALK tyrosine kinase receptor,CRIZOTINIB,CHEMBL601719,Small molecule,INHIBITOR,ALK tyrosine kinase receptor inhibitor,Approved,4.0,2011.0
7,ALK,ALK,ALK tyrosine kinase receptor,CHEMBL4247,ALK tyrosine kinase receptor,ENSARTINIB,CHEMBL4113131,Small molecule,INHIBITOR,ALK tyrosine kinase receptor inhibitor,Approved,4.0,2024.0
8,ALK,ALK,ALK tyrosine kinase receptor,CHEMBL4247,ALK tyrosine kinase receptor,ENTRECTINIB,CHEMBL1983268,Small molecule,INHIBITOR,ALK tyrosine kinase receptor inhibitor,Approved,4.0,2019.0
9,ALK,ALK,ALK tyrosine kinase receptor,CHEMBL4247,ALK tyrosine kinase receptor,LORLATINIB,CHEMBL3286830,Small molecule,INHIBITOR,ALK tyrosine kinase receptor inhibitor,Approved,4.0,2018.0


## 3. Select Drugs For Exploration

For this first multi-target PubMed pass, we search a limited number of drugs per target.

Selection strategy:

1. Prefer approved drugs.
2. Then include highest phase investigational drugs.
3. Limit each target to `MAX_DRUGS_PER_TARGET` to avoid unnecessary API load during exploration.

This gives us enough evidence to validate the process before scaling.

In [3]:
MAX_DRUGS_PER_TARGET = 8


def phase_value(value):
    try:
        return float(value)
    except (TypeError, ValueError):
        return 0.0


selection_df = recommendations_df.copy()
selection_df["phase_sort"] = selection_df["max_phase"].apply(phase_value)
selection_df["is_approved"] = selection_df["approval_status"].eq("Approved")

selection_df = selection_df.sort_values(
    ["target_symbol", "is_approved", "phase_sort", "drug_name"],
    ascending=[True, False, False, True],
)

drugs_to_search_df = (
    selection_df
    .groupby("target_symbol", group_keys=False)
    .head(MAX_DRUGS_PER_TARGET)
    .reset_index(drop=True)
)

print("Drug-target pairs selected:", len(drugs_to_search_df))
print(drugs_to_search_df.groupby("target_symbol").size().to_string())

display(
    drugs_to_search_df[
        [
            "target_symbol",
            "target_display_name",
            "drug_name",
            "approval_status",
            "max_phase",
            "mechanism_of_action",
        ]
    ]
)

Drug-target pairs selected: 58
target_symbol
ALK       8
BRAF      8
EGFR      8
ERBB2     8
KRAS      2
MET       8
PIK3CA    8
VEGFA     8


,target_symbol,target_display_name,drug_name,approval_status,max_phase,mechanism_of_action
0,ALK,ALK,ALECTINIB HYDROCHLORIDE,Approved,4.0,ALK tyrosine kinase receptor inhibitor
1,ALK,ALK,BRIGATINIB,Approved,4.0,ALK tyrosine kinase receptor inhibitor
2,ALK,ALK,CERITINIB,Approved,4.0,ALK tyrosine kinase receptor inhibitor
3,ALK,ALK,CRIZOTINIB,Approved,4.0,ALK tyrosine kinase receptor inhibitor
4,ALK,ALK,ENSARTINIB,Approved,4.0,ALK tyrosine kinase receptor inhibitor
5,ALK,ALK,ENTRECTINIB,Approved,4.0,ALK tyrosine kinase receptor inhibitor
6,ALK,ALK,LORLATINIB,Approved,4.0,ALK tyrosine kinase receptor inhibitor
7,ALK,ALK,ASP-3026,Investigational (Phase 1),1.0,ALK tyrosine kinase receptor inhibitor
8,BRAF,BRAF,DABRAFENIB MESYLATE,Approved,4.0,Serine/threonine-protein kinase B-raf inhibitor
9,BRAF,BRAF,ENCORAFENIB,Approved,4.0,Serine/threonine-protein kinase B-raf inhibitor


## 4. PubMed Helper Functions

In [4]:
PUBMED_SEARCH_URL = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"
PUBMED_SUMMARY_URL = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esummary.fcgi"


def slug(value: str) -> str:
    return re.sub(r"[^a-z0-9]+", "_", str(value).lower()).strip("_")


def save_json(path: Path, data) -> None:
    with path.open("w") as f:
        json.dump(data, f, indent=2)


def pubmed_get(url, params, retries=4, pause=1.0):
    """GET JSON from NCBI E-utilities with simple retries."""
    for attempt in range(1, retries + 1):
        try:
            response = requests.get(url, params=params, timeout=(10, 60))
            response.raise_for_status()
            return response.json()
        except requests.exceptions.RequestException as error:
            print(f"Attempt {attempt}/{retries} failed: {error}")
            if attempt == retries:
                raise
            time.sleep(pause * attempt)


def clean_drug_query_name(drug_name: str) -> str:
    """Keep the ChEMBL name but remove common salt words for broader PubMed matching."""
    value = str(drug_name)
    value = re.sub(r"\b(HYDROCHLORIDE|MESYLATE|SUCCINATE|ANHYDROUS|DIHYDROCHLORIDE)\b", "", value, flags=re.I)
    value = re.sub(r"\s+", " ", value).strip()
    return value or str(drug_name)


def build_query(row) -> str:
    drug = clean_drug_query_name(row["drug_name"])
    target_symbol = str(row["target_symbol"])
    target_display = str(row["target_display_name"])
    target_full = str(row.get("target_full_name", ""))

    target_terms = [target_symbol]
    if target_display and target_display != target_symbol:
        target_terms.append(target_display)
    if target_full and len(target_full) < 90:
        target_terms.append(target_full)

    target_query = " OR ".join(f'"{term}"[Title/Abstract]' for term in dict.fromkeys(target_terms))
    return f'"{drug}"[Title/Abstract] AND ({target_query})'

## 5. Test One Query

In [5]:
test_row = drugs_to_search_df.iloc[0]
test_query = build_query(test_row)

params = {
    "db": "pubmed",
    "term": test_query,
    "retmode": "json",
    "retmax": 5,
    "sort": "relevance",
}

test_search_data = pubmed_get(PUBMED_SEARCH_URL, params)
test_pmids = test_search_data.get("esearchresult", {}).get("idlist", [])
test_count = test_search_data.get("esearchresult", {}).get("count", "0")

print("Target:", test_row["target_symbol"])
print("Drug:", test_row["drug_name"])
print("Query:", test_query)
print("Total matches:", test_count)
print("Fetched PMIDs:", test_pmids)

Target: ALK
Drug: ALECTINIB HYDROCHLORIDE
Query: "ALECTINIB"[Title/Abstract] AND ("ALK"[Title/Abstract] OR "ALK tyrosine kinase receptor"[Title/Abstract])
Total matches: 1279
Fetched PMIDs: ['28586279', '32418886', '39368244', '30069772', '28501140']


## 6. Search PubMed For Selected Drug-Target Pairs

In [6]:
RETMAX = 5
NCBI_PAUSE_SECONDS = 0.35

search_records = []
raw_search_payload = []

for _, row in drugs_to_search_df.iterrows():
    query = build_query(row)
    params = {
        "db": "pubmed",
        "term": query,
        "retmode": "json",
        "retmax": RETMAX,
        "sort": "relevance",
    }

    search_data = pubmed_get(PUBMED_SEARCH_URL, params)
    result = search_data.get("esearchresult", {})
    pmids = result.get("idlist", [])
    total_matches = int(result.get("count", 0))

    record = {
        "target_symbol": row["target_symbol"],
        "target_display_name": row["target_display_name"],
        "target_full_name": row["target_full_name"],
        "drug_name": row["drug_name"],
        "molecule_chembl_id": row["molecule_chembl_id"],
        "approval_status": row["approval_status"],
        "query": query,
        "pmids": pmids,
        "total_matches": total_matches,
        "pubmed_count": len(pmids),
    }
    search_records.append(record)

    raw_search_payload.append(
        {
            "target_symbol": row["target_symbol"],
            "drug_name": row["drug_name"],
            "query": query,
            "response": search_data,
        }
    )

    print(f"{row['target_symbol']} | {row['drug_name']}: {total_matches} total, fetched {len(pmids)}")
    time.sleep(NCBI_PAUSE_SECONDS)

raw_search_file = RAW_DIR / "multi_target_pubmed_search_raw.json"
save_json(raw_search_file, raw_search_payload)

pubmed_search_df = pd.DataFrame(search_records)
print("Saved raw search:", raw_search_file)

display(pubmed_search_df.head(20))

ALK | ALECTINIB HYDROCHLORIDE: 1279 total, fetched 5
ALK | BRIGATINIB: 440 total, fetched 5
ALK | CERITINIB: 642 total, fetched 5
ALK | CRIZOTINIB: 2429 total, fetched 5
ALK | ENSARTINIB: 150 total, fetched 5
ALK | ENTRECTINIB: 118 total, fetched 5
ALK | LORLATINIB: 673 total, fetched 5
ALK | ASP-3026: 0 total, fetched 0
BRAF | DABRAFENIB MESYLATE: 1764 total, fetched 5
BRAF | ENCORAFENIB: 458 total, fetched 5
BRAF | REGORAFENIB: 126 total, fetched 5
BRAF | SORAFENIB TOSYLATE: 6 total, fetched 5
BRAF | VEMURAFENIB: 2208 total, fetched 5
BRAF | CEP-32496: 5 total, fetched 5
BRAF | LIFIRAFENIB: 4 total, fetched 4
BRAF | PLIXORAFENIB: 2 total, fetched 2
EGFR | AFATINIB DIMALEATE: 2 total, fetched 2
EGFR | AMIVANTAMAB: 314 total, fetched 5
EGFR | BRIGATINIB: 94 total, fetched 5
EGFR | CETUXIMAB: 4641 total, fetched 5
EGFR | DACOMITINIB: 352 total, fetched 5
EGFR | DACOMITINIB ANHYDROUS: 352 total, fetched 5
EGFR | ERLOTINIB HYDROCHLORIDE: 5679 total, fetched 5
EGFR | GEFITINIB: 6192 total,

,target_symbol,target_display_name,target_full_name,drug_name,molecule_chembl_id,approval_status,query,pmids,total_matches,pubmed_count
0,ALK,ALK,ALK tyrosine kinase receptor,ALECTINIB HYDROCHLORIDE,CHEMBL3707320,Approved,"""ALECTINIB""[Title/Abstract] AND (""ALK""[Title/A...","[28586279, 32418886, 39368244, 30069772, 28501...",1279,5
1,ALK,ALK,ALK tyrosine kinase receptor,BRIGATINIB,CHEMBL3545311,Approved,"""BRIGATINIB""[Title/Abstract] AND (""ALK""[Title/...","[34537440, 30280657, 39368244, 38835344, 39197...",440,5
2,ALK,ALK,ALK tyrosine kinase receptor,CERITINIB,CHEMBL2403108,Approved,"""CERITINIB""[Title/Abstract] AND (""ALK""[Title/A...","[27573755, 27432227, 38835344, 28126333, 38331...",642,5
3,ALK,ALK,ALK tyrosine kinase receptor,CRIZOTINIB,CHEMBL601719,Approved,"""CRIZOTINIB""[Title/Abstract] AND (""ALK""[Title/...","[34537440, 30280657, 30069772, 24756793, 30069...",2429,5
4,ALK,ALK,ALK tyrosine kinase receptor,ENSARTINIB,CHEMBL4113131,Approved,"""ENSARTINIB""[Title/Abstract] AND (""ALK""[Title/...","[34473194, 38331773, 40310967, 36948240, 31857...",150,5
5,ALK,ALK,ALK tyrosine kinase receptor,ENTRECTINIB,CHEMBL1983268,Approved,"""ENTRECTINIB""[Title/Abstract] AND (""ALK""[Title...","[28183697, 31372957, 30050303, 30425456, 41091...",118,5
6,ALK,ALK,ALK tyrosine kinase receptor,LORLATINIB,CHEMBL3286830,Approved,"""LORLATINIB""[Title/Abstract] AND (""ALK""[Title/...","[38819031, 33207094, 35534623, 38554546, 39368...",673,5
7,ALK,ALK,ALK tyrosine kinase receptor,ASP-3026,CHEMBL3545360,Investigational (Phase 1),"""ASP-3026""[Title/Abstract] AND (""ALK""[Title/Ab...",[],0,0
8,BRAF,BRAF,Serine/threonine-protein kinase B-raf,DABRAFENIB MESYLATE,CHEMBL2105729,Approved,"""DABRAFENIB""[Title/Abstract] AND (""BRAF""[Title...","[38278874, 38899716, 31231568, 37733309, 28891...",1764,5
9,BRAF,BRAF,Serine/threonine-protein kinase B-raf,ENCORAFENIB,CHEMBL3301612,Approved,"""ENCORAFENIB""[Title/Abstract] AND (""BRAF""[Titl...","[40444708, 39863775, 31566309, 33503393, 31231...",458,5


## 7. Fetch Article Summaries

In [7]:
article_records = []
raw_summary_payload = []

for record in search_records:
    pmids = record["pmids"]
    if not pmids:
        continue

    params = {
        "db": "pubmed",
        "id": ",".join(pmids),
        "retmode": "json",
    }

    summary_data = pubmed_get(PUBMED_SUMMARY_URL, params)
    raw_summary_payload.append(
        {
            "target_symbol": record["target_symbol"],
            "drug_name": record["drug_name"],
            "pmids": pmids,
            "response": summary_data,
        }
    )

    result_data = summary_data.get("result", {})

    for rank, pmid in enumerate(pmids, start=1):
        article = result_data.get(pmid, {})
        article_records.append(
            {
                "target_symbol": record["target_symbol"],
                "target_display_name": record["target_display_name"],
                "target_full_name": record["target_full_name"],
                "drug_name": record["drug_name"],
                "molecule_chembl_id": record["molecule_chembl_id"],
                "approval_status": record["approval_status"],
                "pmid": pmid,
                "result_rank": rank,
                "title": article.get("title"),
                "journal": article.get("fulljournalname"),
                "publication_date": article.get("pubdate"),
                "authors": " | ".join(author.get("name", "") for author in article.get("authors", [])[:5]),
                "query": record["query"],
                "url": f"https://pubmed.ncbi.nlm.nih.gov/{pmid}/",
                "source": "PubMed",
            }
        )

    print(f"Fetched summaries for {record['target_symbol']} | {record['drug_name']}: {len(pmids)} PMIDs")
    time.sleep(NCBI_PAUSE_SECONDS)

raw_summary_file = RAW_DIR / "multi_target_pubmed_summary_raw.json"
save_json(raw_summary_file, raw_summary_payload)

pubmed_evidence_df = pd.DataFrame(article_records)

print("Saved raw summaries:", raw_summary_file)
print("Evidence rows:", len(pubmed_evidence_df))

display(pubmed_evidence_df.head(20))

Fetched summaries for ALK | ALECTINIB HYDROCHLORIDE: 5 PMIDs
Fetched summaries for ALK | BRIGATINIB: 5 PMIDs
Fetched summaries for ALK | CERITINIB: 5 PMIDs
Fetched summaries for ALK | CRIZOTINIB: 5 PMIDs
Fetched summaries for ALK | ENSARTINIB: 5 PMIDs
Fetched summaries for ALK | ENTRECTINIB: 5 PMIDs
Fetched summaries for ALK | LORLATINIB: 5 PMIDs
Fetched summaries for BRAF | DABRAFENIB MESYLATE: 5 PMIDs
Fetched summaries for BRAF | ENCORAFENIB: 5 PMIDs
Fetched summaries for BRAF | REGORAFENIB: 5 PMIDs
Fetched summaries for BRAF | SORAFENIB TOSYLATE: 5 PMIDs
Fetched summaries for BRAF | VEMURAFENIB: 5 PMIDs
Fetched summaries for BRAF | CEP-32496: 5 PMIDs
Fetched summaries for BRAF | LIFIRAFENIB: 4 PMIDs
Fetched summaries for BRAF | PLIXORAFENIB: 2 PMIDs
Fetched summaries for EGFR | AFATINIB DIMALEATE: 2 PMIDs
Fetched summaries for EGFR | AMIVANTAMAB: 5 PMIDs
Fetched summaries for EGFR | BRIGATINIB: 5 PMIDs
Fetched summaries for EGFR | CETUXIMAB: 5 PMIDs
Fetched summaries for EGFR | DACO

,target_symbol,target_display_name,target_full_name,drug_name,molecule_chembl_id,approval_status,pmid,result_rank,title,journal,publication_date,authors,query,url,source
0,ALK,ALK,ALK tyrosine kinase receptor,ALECTINIB HYDROCHLORIDE,CHEMBL3707320,Approved,28586279,1,Alectinib versus Crizotinib in Untreated ALK-P...,The New England journal of medicine,2017 Aug 31,Peters S | Camidge DR | Shaw AT | Gadgeel S | ...,"""ALECTINIB""[Title/Abstract] AND (""ALK""[Title/A...",https://pubmed.ncbi.nlm.nih.gov/28586279/,PubMed
1,ALK,ALK,ALK tyrosine kinase receptor,ALECTINIB HYDROCHLORIDE,CHEMBL3707320,Approved,32418886,2,Updated overall survival and final progression...,Annals of oncology : official journal of the E...,2020 Aug,Mok T | Camidge DR | Gadgeel SM | Rosell R | D...,"""ALECTINIB""[Title/Abstract] AND (""ALK""[Title/A...",https://pubmed.ncbi.nlm.nih.gov/32418886/,PubMed
2,ALK,ALK,ALK tyrosine kinase receptor,ALECTINIB HYDROCHLORIDE,CHEMBL3707320,Approved,39368244,3,Systematic review and network meta-analysis of...,"Lung cancer (Amsterdam, Netherlands)",2024 Nov,Ou SH | Kilvert H | Candlish J | Lee B | Polli A,"""ALECTINIB""[Title/Abstract] AND (""ALK""[Title/A...",https://pubmed.ncbi.nlm.nih.gov/39368244/,PubMed
3,ALK,ALK,ALK tyrosine kinase receptor,ALECTINIB HYDROCHLORIDE,CHEMBL3707320,Approved,30069772,4,Alectinib.,Recent results in cancer research. Fortschritt...,2018,Herden M | Waller CF,"""ALECTINIB""[Title/Abstract] AND (""ALK""[Title/A...",https://pubmed.ncbi.nlm.nih.gov/30069772/,PubMed
4,ALK,ALK,ALK tyrosine kinase receptor,ALECTINIB HYDROCHLORIDE,CHEMBL3707320,Approved,28501140,5,Alectinib versus crizotinib in patients with A...,"Lancet (London, England)",2017 Jul 1,Hida T | Nokihara H | Kondo M | Kim YH | Azuma K,"""ALECTINIB""[Title/Abstract] AND (""ALK""[Title/A...",https://pubmed.ncbi.nlm.nih.gov/28501140/,PubMed
5,ALK,ALK,ALK tyrosine kinase receptor,BRIGATINIB,CHEMBL3545311,Approved,34537440,1,Brigatinib Versus Crizotinib in ALK Inhibitor-...,Journal of thoracic oncology : official public...,2021 Dec,Camidge DR | Kim HR | Ahn MJ | Yang JCH | Han JY,"""BRIGATINIB""[Title/Abstract] AND (""ALK""[Title/...",https://pubmed.ncbi.nlm.nih.gov/34537440/,PubMed
6,ALK,ALK,ALK tyrosine kinase receptor,BRIGATINIB,CHEMBL3545311,Approved,30280657,2,Brigatinib versus Crizotinib in ALK-Positive N...,The New England journal of medicine,2018 Nov 22,Camidge DR | Kim HR | Ahn MJ | Yang JC | Han JY,"""BRIGATINIB""[Title/Abstract] AND (""ALK""[Title/...",https://pubmed.ncbi.nlm.nih.gov/30280657/,PubMed
7,ALK,ALK,ALK tyrosine kinase receptor,BRIGATINIB,CHEMBL3545311,Approved,39368244,3,Systematic review and network meta-analysis of...,"Lung cancer (Amsterdam, Netherlands)",2024 Nov,Ou SH | Kilvert H | Candlish J | Lee B | Polli A,"""BRIGATINIB""[Title/Abstract] AND (""ALK""[Title/...",https://pubmed.ncbi.nlm.nih.gov/39368244/,PubMed
8,ALK,ALK,ALK tyrosine kinase receptor,BRIGATINIB,CHEMBL3545311,Approved,38835344,4,ALK inhibitors in cancer: mechanisms of resist...,"Cancer drug resistance (Alhambra, Calif.)",2024,Poei D | Ali S | Ye S | Hsu R,"""BRIGATINIB""[Title/Abstract] AND (""ALK""[Title/...",https://pubmed.ncbi.nlm.nih.gov/38835344/,PubMed
9,ALK,ALK,ALK tyrosine kinase receptor,BRIGATINIB,CHEMBL3545311,Approved,39197358,5,Real-world treatment sequencing and effectiven...,"Lung cancer (Amsterdam, Netherlands)",2024 Sep,Bauman JR | Liu G | Preeshagul I | Liu SV | Me...,"""BRIGATINIB""[Title/Abstract] AND (""ALK""[Title/...",https://pubmed.ncbi.nlm.nih.gov/39197358/,PubMed


## 8. Save Processed PubMed Evidence

In [9]:
pubmed_evidence_file = PROCESSED_DIR / "multi_target_pubmed_evidence.csv"
pubmed_evidence_df.to_csv(pubmed_evidence_file, index=False)

print("Saved:", pubmed_evidence_file)
print("Rows:", len(pubmed_evidence_df))

Saved: /Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/ai-precision-medicine-lab/therapeutic-strategy-assistant/data/processed/multi_target_pubmed_evidence.csv
Rows: 234


## 9. Build PubMed Summary

In [10]:
summary_columns = [
    "target_symbol",
    "target_display_name",
    "drug_name",
    "molecule_chembl_id",
    "pubmed_count",
    "top_pubmed_titles",
]

if pubmed_evidence_df.empty:
    pubmed_summary_df = pd.DataFrame(columns=summary_columns)
else:
    pubmed_summary_df = (
        pubmed_evidence_df
        .groupby(["target_symbol", "target_display_name", "drug_name", "molecule_chembl_id"], dropna=False)
        .agg(
            pubmed_count=("pmid", "nunique"),
            top_pubmed_titles=("title", lambda values: " | ".join(list(values.dropna())[:3])),
        )
        .reset_index()
    )

pubmed_summary_file = PROCESSED_DIR / "multi_target_pubmed_summary.csv"
pubmed_summary_df.to_csv(pubmed_summary_file, index=False)

print("Saved:", pubmed_summary_file)
print("Rows:", len(pubmed_summary_df))

display(pubmed_summary_df.head(30))

Saved: /Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/ai-precision-medicine-lab/therapeutic-strategy-assistant/data/processed/multi_target_pubmed_summary.csv
Rows: 53


,target_symbol,target_display_name,drug_name,molecule_chembl_id,pubmed_count,top_pubmed_titles
0,ALK,ALK,ALECTINIB HYDROCHLORIDE,CHEMBL3707320,5,Alectinib versus Crizotinib in Untreated ALK-P...
1,ALK,ALK,BRIGATINIB,CHEMBL3545311,5,Brigatinib Versus Crizotinib in ALK Inhibitor-...
2,ALK,ALK,CERITINIB,CHEMBL2403108,5,The role of the ALK receptor in cancer biology...
3,ALK,ALK,CRIZOTINIB,CHEMBL601719,5,Brigatinib Versus Crizotinib in ALK Inhibitor-...
4,ALK,ALK,ENSARTINIB,CHEMBL4113131,5,Ensartinib vs Crizotinib for Patients With Ana...
5,ALK,ALK,ENTRECTINIB,CHEMBL1983268,5,Safety and Antitumor Activity of the Multitarg...
6,ALK,ALK,LORLATINIB,CHEMBL3286830,5,Lorlatinib Versus Crizotinib in Patients With ...
7,BRAF,BRAF,CEP-32496,CHEMBL2029988,5,"[(11)C-carbonyl]CEP-32496: radiosynthesis, bio..."
8,BRAF,BRAF,DABRAFENIB MESYLATE,CHEMBL2105729,5,BRAF - a tumour-agnostic drug target with line...
9,BRAF,BRAF,ENCORAFENIB,CHEMBL3301612,5,"Encorafenib, Cetuximab, and mFOLFOX6 in BRAF-M..."


## 10. Coverage Summary

In [11]:
coverage_rows = []

for target_symbol, target_df in drugs_to_search_df.groupby("target_symbol"):
    evidence_target_df = pubmed_evidence_df[pubmed_evidence_df["target_symbol"] == target_symbol]
    searched_count = target_df["drug_name"].nunique()
    matched_count = evidence_target_df["drug_name"].nunique() if not evidence_target_df.empty else 0

    coverage_rows.append(
        {
            "target_symbol": target_symbol,
            "target_display_name": target_df["target_display_name"].iloc[0],
            "source": "PubMed",
            "drug_pairs_searched": int(searched_count),
            "drugs_with_pubmed_evidence": int(matched_count),
            "pubmed_evidence_rows": int(len(evidence_target_df)),
            "raw_saved": True,
            "processed_saved": True,
            "status": "working" if matched_count > 0 else "needs_review",
            "notes": "" if matched_count > 0 else "No PubMed title/abstract matches found for selected drugs.",
        }
    )

pubmed_coverage_df = pd.DataFrame(coverage_rows)

pubmed_coverage_file = PROCESSED_DIR / "multi_target_pubmed_coverage_summary.csv"
pubmed_coverage_df.to_csv(pubmed_coverage_file, index=False)

print("Saved:", pubmed_coverage_file)
display(pubmed_coverage_df)

Saved: /Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/ai-precision-medicine-lab/therapeutic-strategy-assistant/data/processed/multi_target_pubmed_coverage_summary.csv


,target_symbol,target_display_name,source,drug_pairs_searched,drugs_with_pubmed_evidence,pubmed_evidence_rows,raw_saved,processed_saved,status,notes
0,ALK,ALK,PubMed,8,7,35,True,True,working,
1,BRAF,BRAF,PubMed,8,8,36,True,True,working,
2,EGFR,EGFR,PubMed,8,8,37,True,True,working,
3,ERBB2,HER2,PubMed,8,8,29,True,True,working,
4,KRAS,KRAS,PubMed,2,2,10,True,True,working,
5,MET,MET,PubMed,8,7,31,True,True,working,
6,PIK3CA,PIK3CA,PubMed,8,6,23,True,True,working,
7,VEGFA,VEGFA,PubMed,8,7,33,True,True,working,


## 11. Final Summary

In [12]:
print("Multi-Target PubMed Exploration Complete")
print("=" * 80)
print("Drug-target pairs searched:", len(drugs_to_search_df))
print("Evidence rows:", len(pubmed_evidence_df))
print("Summary rows:", len(pubmed_summary_df))

print("\nEvidence rows by target:")
if not pubmed_evidence_df.empty:
    print(pubmed_evidence_df.groupby("target_symbol").size().sort_values(ascending=False).to_string())
else:
    print("No PubMed evidence rows found.")

print("\nFiles created:")
for path in [
    raw_search_file,
    raw_summary_file,
    pubmed_evidence_file,
    pubmed_summary_file,
    pubmed_coverage_file,
]:
    print("-", path)

display(pubmed_coverage_df)

Multi-Target PubMed Exploration Complete
Drug-target pairs searched: 58
Evidence rows: 234
Summary rows: 53

Evidence rows by target:
target_symbol
EGFR      37
BRAF      36
ALK       35
VEGFA     33
MET       31
ERBB2     29
PIK3CA    23
KRAS      10

Files created:
- /Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/ai-precision-medicine-lab/therapeutic-strategy-assistant/data/raw/pubmed/multi_target_pubmed_search_raw.json
- /Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/ai-precision-medicine-lab/therapeutic-strategy-assistant/data/raw/pubmed/multi_target_pubmed_summary_raw.json
- /Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/ai-precision-medicine-lab/therapeutic-strategy-assistant/data/processed/multi_target_pubmed_evidence.csv
- /Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/ai-precision-medicine-lab/therapeutic-strategy-assistant/data/processed/multi_target_pubmed_summary.csv
- /Users/daniel/Docu

,target_symbol,target_display_name,source,drug_pairs_searched,drugs_with_pubmed_evidence,pubmed_evidence_rows,raw_saved,processed_saved,status,notes
0,ALK,ALK,PubMed,8,7,35,True,True,working,
1,BRAF,BRAF,PubMed,8,8,36,True,True,working,
2,EGFR,EGFR,PubMed,8,8,37,True,True,working,
3,ERBB2,HER2,PubMed,8,8,29,True,True,working,
4,KRAS,KRAS,PubMed,2,2,10,True,True,working,
5,MET,MET,PubMed,8,7,31,True,True,working,
6,PIK3CA,PIK3CA,PubMed,8,6,23,True,True,working,
7,VEGFA,VEGFA,PubMed,8,7,33,True,True,working,
